# EMT three-phase inverter state-space extraction

This notebook demonstrates state-space extraction and frame-aware modal analysis for a three-phase averaged voltage-source inverter connected to an ideal voltage source through separate three-phase resistor and inductor components.

Topology:

```text
GND -- Inverter -- nInv -- R -- nMid -- L -- nGrid -- VoltageSource -- GND
```

The notebook performs three checks:

1. **Check A: native extraction verification**
   - Compares DPsim-extracted eigenvalues with an analytical mixed-frame model written in the same DPsim-native coordinates.
   - Includes numerical assertions.

2. **Check B: stability-classification consistency**
   - Compares the stability conclusions obtained from:
     - DPsim `GlobalDQ0` one-step discrete eigenvalues,
     - native-frame Floquet multipliers over one fundamental period,
     - analytical continuous-time global dq0 eigenvalues.
   - Only the stability classification is asserted.

3. **Check C: modal and time-domain demonstration**
   - Visualizes the stable and unstable cases close to the stability boundary using:
     - DPsim native-frame Floquet multipliers,
     - DPsim `GlobalDQ0` discrete eigenvalues,
     - DPsim `GlobalDQ0` continuous eigenvalues calculated from the discrete eigenvalues,
     - analytical continuous-time global dq0 eigenvalues,
     - DPsim EMT active-power response.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import dpsimpy

emt = dpsimpy.emt
ph3 = emt.ph3

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
Solver = dpsimpy.Solver
StateSpaceModalAnalysis = dpsimpy.StateSpaceModalAnalysis
StateSpaceAnalysisFrame = dpsimpy.StateSpaceAnalysisFrame

## Parameters

In [2]:
# Simulation settings
time_step = 50e-6
final_time_short = time_step
final_time_validation = 1.0

# System
system_frequency = 50.0
omega0 = 2.0 * np.pi * system_frequency

# Fixed ideal source voltage
grid_voltage_rms_ll = 400.0
source_phase_deg = -90.0

# Grid branch
r_grid = 0.3
l_grid = 1.0e-3

# Inverter filter
lf = 2.0e-3
cf = 10.0e-6
rf = 0.2
rc = 0.2

# PLL
kp_pll = 0.25
ki_pll = 0.2
omega_cutoff = omega0

# Power references
p_ref = 10000.0
q_ref = 5000.0

# Base power controller
kp_power_ctrl_base = 0.05
ki_power_ctrl_base = 0.2

# Base current controller
kp_curr_ctrl_base = 0.25
ki_curr_ctrl_base = 1.0

# Gain scales close to the stability boundary
gain_scale_stable = 3.7
gain_scale_unstable = 3.8

sim_name_base = "EMT_Ph3_Inverter_StateSpaceExtraction"

## Common utilities

In [3]:
abc_scale = np.sqrt(2.0 / 3.0)


def abc_phasor(phase_a_phasor):
    return np.array(
        [
            [phase_a_phasor],
            [phase_a_phasor * np.exp(-1j * 2.0 * np.pi / 3.0)],
            [phase_a_phasor * np.exp(+1j * 2.0 * np.pi / 3.0)],
        ],
        dtype=np.complex128,
    )


def abc_instantaneous_from_phasor(phase_a_phasor, t=0.0):
    return np.real(
        abc_scale * abc_phasor(phase_a_phasor).reshape(3) * np.exp(1j * omega0 * t)
    )


def three_phase_param(value):
    return dpsimpy.Math.single_phase_parameter_to_three_phase(value)


def source_phasor():
    return grid_voltage_rms_ll * np.exp(1j * np.deg2rad(source_phase_deg))


def park_matrix(theta):
    k = np.sqrt(2.0 / 3.0)

    return np.array(
        [
            [
                k * np.cos(theta),
                k * np.cos(theta - 2.0 * np.pi / 3.0),
                k * np.cos(theta + 2.0 * np.pi / 3.0),
            ],
            [
                -k * np.sin(theta),
                -k * np.sin(theta - 2.0 * np.pi / 3.0),
                -k * np.sin(theta + 2.0 * np.pi / 3.0),
            ],
        ],
        dtype=float,
    )


def inv_park_matrix(theta):
    k = np.sqrt(2.0 / 3.0)

    return np.array(
        [
            [k * np.cos(theta), -k * np.sin(theta)],
            [
                k * np.cos(theta - 2.0 * np.pi / 3.0),
                -k * np.sin(theta - 2.0 * np.pi / 3.0),
            ],
            [
                k * np.cos(theta + 2.0 * np.pi / 3.0),
                -k * np.sin(theta + 2.0 * np.pi / 3.0),
            ],
        ],
        dtype=float,
    )


def park_matrix_3(theta):
    k = np.sqrt(2.0 / 3.0)
    k0 = 1.0 / np.sqrt(3.0)

    return np.array(
        [
            [
                k * np.cos(theta),
                k * np.cos(theta - 2.0 * np.pi / 3.0),
                k * np.cos(theta + 2.0 * np.pi / 3.0),
            ],
            [
                -k * np.sin(theta),
                -k * np.sin(theta - 2.0 * np.pi / 3.0),
                -k * np.sin(theta + 2.0 * np.pi / 3.0),
            ],
            [k0, k0, k0],
        ],
        dtype=float,
    )


def park_matrix_3_derivative(theta):
    T = park_matrix_3(theta)

    dT = np.zeros((3, 3))
    dT[0, :] = T[1, :]
    dT[1, :] = -T[0, :]

    return dT


def numerical_jacobian(f, x0, eps=1e-6):
    x0 = np.asarray(x0, dtype=float)
    f0 = np.asarray(f(x0), dtype=float)
    A = np.zeros((len(f0), len(x0)))

    for k in range(len(x0)):
        h = eps * max(1.0, abs(x0[k]))

        xp = x0.copy()
        xm = x0.copy()

        xp[k] += h
        xm[k] -= h

        A[:, k] = (f(xp) - f(xm)) / (2.0 * h)

    return A


def trapezoidal_discretization(A, dt):
    I = np.eye(A.shape[0])

    return np.linalg.solve(
        I - 0.5 * dt * A,
        I + 0.5 * dt * A,
    )


def bilinear_lambda_from_z(z, dt):
    return (2.0 / dt) * ((z - 1.0) / (z + 1.0))


def finite_complex(values):
    values = np.asarray(values).reshape(-1)

    return values[np.isfinite(values.real) & np.isfinite(values.imag)]


def closest_eigenvalue_matches(reference, candidate):
    rows = []

    for value_ref in reference:
        idx = np.argmin(np.abs(candidate - value_ref))
        rows.append(
            {
                "reference": value_ref,
                "closest": candidate[idx],
                "distance": abs(candidate[idx] - value_ref),
            }
        )

    return sorted(rows, key=lambda row: row["distance"], reverse=True)


def symmetric_max_eigenvalue_distance(a, b):
    a = finite_complex(a)
    b = finite_complex(b)

    return max(
        closest_eigenvalue_matches(a, b)[0]["distance"],
        closest_eigenvalue_matches(b, a)[0]["distance"],
    )


def eigenvalue_table(lambdas, source, n=None):
    values = finite_complex(lambdas)
    order = np.argsort(values.real)[::-1]

    if n is not None:
        order = order[:n]

    values = values[order]

    return pd.DataFrame(
        {
            "source": source,
            "Re(lambda) [1/s]": values.real,
            "Im(lambda) [rad/s]": values.imag,
            "frequency [Hz]": np.abs(values.imag) / (2.0 * np.pi),
        }
    )

## DPsim system setup

The operating point is computed for a fixed source voltage. The inverter power references are interpreted at the inverter capacitor voltage.

In [4]:
def steady_state_operating_point(gain_scale=1.0):
    kp_power_ctrl = gain_scale * kp_power_ctrl_base
    ki_power_ctrl = gain_scale * ki_power_ctrl_base
    kp_curr_ctrl = gain_scale * kp_curr_ctrl_base
    ki_curr_ctrl = gain_scale * ki_curr_ctrl_base

    S = p_ref + 1j * q_ref
    E = source_phasor()

    Z_total = rc + r_grid + 1j * omega0 * l_grid
    C = Z_total * np.conj(S)

    roots = np.roots(
        [
            1.0,
            -(abs(E) ** 2 + 2.0 * C.real),
            abs(C) ** 2,
        ]
    )

    roots_real = np.real(roots[np.isreal(roots)])
    y = np.max(roots_real[roots_real > 0.0])

    Vc = (y - np.conj(C)) / np.conj(E)
    Iinj = np.conj(S / Vc)

    Uinv = Vc - rc * Iinj
    Umid = E + 1j * omega0 * l_grid * Iinj

    If = Iinj + 1j * omega0 * cf * Vc
    Vref = Vc + (rf + 1j * omega0 * lf) * If

    theta = np.angle(Vc)

    vc_abc = abc_instantaneous_from_phasor(Vc)
    if_abc = abc_instantaneous_from_phasor(If)
    i_line_abc = abc_instantaneous_from_phasor(Iinj)
    u_inv_abc = abc_instantaneous_from_phasor(Uinv)
    u_mid_abc = abc_instantaneous_from_phasor(Umid)
    e_grid_abc = abc_instantaneous_from_phasor(E)
    v_ref_abc = abc_instantaneous_from_phasor(Vref)

    T = park_matrix(theta)

    vc_dq = T @ vc_abc
    i_dq = T @ i_line_abc
    v_ref_dq = T @ v_ref_abc

    vc_d, vc_q = vc_dq
    i_d, i_q = i_dq

    p0 = vc_d * i_d + vc_q * i_q
    q0 = -vc_d * i_q + vc_q * i_d

    phi_pll = 0.0
    p_filtered = p0
    q_filtered = q0

    phi_d = (i_d - kp_power_ctrl * (p_ref - p_filtered)) / ki_power_ctrl
    phi_q = (i_q - kp_power_ctrl * (q_filtered - q_ref)) / ki_power_ctrl

    i_ref_d = kp_power_ctrl * (p_ref - p_filtered) + ki_power_ctrl * phi_d
    i_ref_q = kp_power_ctrl * (q_filtered - q_ref) + ki_power_ctrl * phi_q

    gamma_d = (v_ref_dq[0] - kp_curr_ctrl * (i_ref_d - i_d)) / ki_curr_ctrl
    gamma_q = (v_ref_dq[1] - kp_curr_ctrl * (i_ref_q - i_q)) / ki_curr_ctrl

    x_inv = np.zeros(14)
    x_inv[0] = theta
    x_inv[1] = phi_pll
    x_inv[2] = p_filtered
    x_inv[3] = q_filtered
    x_inv[4] = phi_d
    x_inv[5] = phi_q
    x_inv[6] = gamma_d
    x_inv[7] = gamma_q
    x_inv[8:11] = vc_abc
    x_inv[11:14] = if_abc

    x_full = np.zeros(17)
    x_full[:14] = x_inv
    x_full[14:17] = i_line_abc

    return {
        "E": E,
        "Vc": Vc,
        "Uinv": Uinv,
        "Umid": Umid,
        "Iinj": Iinj,
        "If": If,
        "Vref": Vref,
        "theta": theta,
        "x_inv": x_inv,
        "x_full": x_full,
        "vc_abc": vc_abc,
        "if_abc": if_abc,
        "i_line_abc": i_line_abc,
        "u_inv_abc": u_inv_abc,
        "u_mid_abc": u_mid_abc,
        "e_grid_abc": e_grid_abc,
    }


def build_dpsim_system(op, gain_scale):
    n_inv = emt.SimNode("nInv", PhaseType.ABC)
    n_mid = emt.SimNode("nMid", PhaseType.ABC)
    n_grid = emt.SimNode("nGrid", PhaseType.ABC)

    n_inv.set_initial_voltage(op["Uinv"])
    n_mid.set_initial_voltage(op["Umid"])
    n_grid.set_initial_voltage(op["E"])

    voltage_source = ph3.VoltageSource("VoltageSource")
    voltage_source.set_parameters(
        abc_phasor(op["E"]),
        system_frequency,
    )

    resistor = ph3.Resistor("GridResistor")
    resistor.set_parameters(three_phase_param(r_grid))

    inductor = ph3.Inductor("GridInductor")
    inductor.set_parameters(three_phase_param(l_grid))

    inverter = ph3.AvVoltSourceInverterStateSpace("INV_SSN_PLL")
    inverter.set_parameters(
        lf,
        cf,
        rf,
        rc,
        omega0,
        kp_pll,
        ki_pll,
        omega_cutoff,
        p_ref,
        q_ref,
        gain_scale * kp_power_ctrl_base,
        gain_scale * ki_power_ctrl_base,
        gain_scale * kp_curr_ctrl_base,
        gain_scale * ki_curr_ctrl_base,
    )

    inverter.connect([emt.SimNode.gnd, n_inv])
    resistor.connect([n_inv, n_mid])
    inductor.connect([n_mid, n_grid])
    voltage_source.connect([emt.SimNode.gnd, n_grid])

    system = SystemTopology(
        system_frequency,
        [n_inv, n_mid, n_grid],
        [inverter, resistor, inductor, voltage_source],
    )

    return system, inverter


def get_native_modal_results(sim):
    extractor = sim.get_state_space_extractor()

    modal = StateSpaceModalAnalysis(extractor)
    modal.update()

    return {
        "Ad": np.array(extractor.get_discrete_state_matrix(), copy=True),
        "z": np.array(modal.get_discrete_eigenvalues(), copy=True).reshape(-1),
        "lambda": np.array(modal.get_continuous_eigenvalues(), copy=True).reshape(-1),
        "state_count": extractor.get_state_count(),
    }


def get_global_dq0_modal_results(sim, theta0=0.0):
    extractor = sim.get_state_space_extractor()

    modal = StateSpaceModalAnalysis(extractor)
    modal.set_analysis_frame(StateSpaceAnalysisFrame.GlobalDQ0)
    modal.set_global_dq0_frame(omega0, theta0)
    modal.update()

    return {
        "z": np.array(modal.get_discrete_eigenvalues(), copy=True).reshape(-1),
        "lambda": np.array(modal.get_continuous_eigenvalues(), copy=True).reshape(-1),
    }


def run_dpsim(gain_scale, final_time, case_name, log_signals):
    sim_name = f"{sim_name_base}_{case_name}_gain_{gain_scale:g}".replace(".", "_")
    Logger.set_log_dir("logs/" + sim_name)

    op = steady_state_operating_point(gain_scale)
    system, inverter = build_dpsim_system(op, gain_scale)

    sim = Simulation(sim_name)
    sim.set_system(system)

    if log_signals:
        logger = Logger(sim_name)
        logger.log_attribute("p_inst", inverter.attr("p_inst"))
        logger.log_attribute("q_inst", inverter.attr("q_inst"))
        logger.log_attribute("vc_d", inverter.attr("vc_d"))
        logger.log_attribute("vc_q", inverter.attr("vc_q"))
        logger.log_attribute("omega_pll", inverter.attr("omega_pll"))
        sim.add_logger(logger)

    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)

    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)
    sim.do_state_space_extraction(True)

    sim.set_time_step(time_step)
    sim.set_final_time(final_time)

    n_steps = int(round(final_time / time_step))
    modal_first = None

    sim.start()

    for step_idx in range(n_steps):
        sim.next()

        if step_idx == 0:
            modal_first = get_native_modal_results(sim)

    sim.stop()

    return {
        "sim_name": sim_name,
        "op": op,
        "modal": modal_first,
    }

## Check A: DPsim native extraction against analytical mixed-frame model

This check verifies the DPsim state-space extraction in the native mixed coordinates used by the extracted model.

The analytical model uses the same state ordering as the extracted DPsim model:

```text
thetaPLL, phiPLL, P, Q, phiD, phiQ, gammaD, gammaQ,
vcA, vcB, vcC, ifA, ifB, ifC,
iLA, iLB, iLC
```

The comparison is performed using discrete eigenvalues `z` and continuous eigenvalues `lambda` calculated from the discrete eigenvalues.

The native-frame eigenvalues are used here only to verify the extracted native EMT state-space realization. Since the native abc/mixed-frame model is periodic in steady state, these instantaneous native-frame eigenvalues are not used as the final stability criterion.

In [ ]:
def inverter_rhs_mixed(x_inv, u_abc, gain_scale):
    kp_power_ctrl = gain_scale * kp_power_ctrl_base
    ki_power_ctrl = gain_scale * ki_power_ctrl_base
    kp_curr_ctrl = gain_scale * kp_curr_ctrl_base
    ki_curr_ctrl = gain_scale * ki_curr_ctrl_base

    theta = x_inv[0]
    phi_pll = x_inv[1]
    p_filtered = x_inv[2]
    q_filtered = x_inv[3]
    phi_d = x_inv[4]
    phi_q = x_inv[5]
    gamma_d = x_inv[6]
    gamma_q = x_inv[7]
    vc_abc = x_inv[8:11]
    if_abc = x_inv[11:14]

    T = park_matrix(theta)
    T_inv = inv_park_matrix(theta)

    i_inj_abc = (vc_abc - u_abc) / rc

    vc_dq = T @ vc_abc
    i_dq = T @ i_inj_abc

    vc_d, vc_q = vc_dq
    i_d, i_q = i_dq

    p_inst = vc_d * i_d + vc_q * i_q
    q_inst = -vc_d * i_q + vc_q * i_d

    omega_pll = omega0 + kp_pll * vc_q + ki_pll * phi_pll

    i_ref_d = kp_power_ctrl * (p_ref - p_filtered) + ki_power_ctrl * phi_d
    i_ref_q = kp_power_ctrl * (q_filtered - q_ref) + ki_power_ctrl * phi_q

    v_ref_d = kp_curr_ctrl * (i_ref_d - i_d) + ki_curr_ctrl * gamma_d
    v_ref_q = kp_curr_ctrl * (i_ref_q - i_q) + ki_curr_ctrl * gamma_q

    v_ref_abc = T_inv @ np.array([v_ref_d, v_ref_q])

    dx = np.zeros(14)

    dx[0] = omega_pll
    dx[1] = vc_q
    dx[2] = omega_cutoff * (p_inst - p_filtered)
    dx[3] = omega_cutoff * (q_inst - q_filtered)
    dx[4] = p_ref - p_filtered
    dx[5] = q_filtered - q_ref
    dx[6] = i_ref_d - i_d
    dx[7] = i_ref_q - i_q
    dx[8:11] = if_abc / cf + (u_abc - vc_abc) / (cf * rc)
    dx[11:14] = (v_ref_abc - vc_abc - rf * if_abc) / lf

    return dx


def full_rhs_mixed(x_full, op, gain_scale):
    x_inv = x_full[:14]
    i_line_abc = x_full[14:17]

    vc_abc = x_inv[8:11]
    u_inv_abc = vc_abc - rc * i_line_abc

    dx_inv = inverter_rhs_mixed(x_inv, u_inv_abc, gain_scale)
    di_line = (u_inv_abc - r_grid * i_line_abc - op["e_grid_abc"]) / l_grid

    dx = np.zeros(17)
    dx[:14] = dx_inv
    dx[14:17] = di_line

    return dx


def analytical_mixed_model(op, gain_scale):
    A = numerical_jacobian(
        lambda x: full_rhs_mixed(x, op, gain_scale),
        op["x_full"],
    )

    Ad = trapezoidal_discretization(A, time_step)
    z = np.linalg.eigvals(Ad)
    lambdas = bilinear_lambda_from_z(z, time_step)

    return {
        "A": A,
        "Ad": Ad,
        "z": z,
        "lambda": lambdas,
    }


gain_scale_check_a = gain_scale_stable

dpsim_check_a = run_dpsim(
    gain_scale=gain_scale_check_a,
    final_time=final_time_short,
    case_name="check_a",
    log_signals=False,
)

op_check_a = dpsim_check_a["op"]
modal_dpsim_check_a = dpsim_check_a["modal"]
analytic_mixed_check_a = analytical_mixed_model(op_check_a, gain_scale_check_a)

z_error_check_a = symmetric_max_eigenvalue_distance(
    modal_dpsim_check_a["z"],
    analytic_mixed_check_a["z"],
)

lambda_error_check_a = symmetric_max_eigenvalue_distance(
    modal_dpsim_check_a["lambda"],
    analytic_mixed_check_a["lambda"],
)

print("Check A: DPsim extracted eigenvalues vs analytical mixed-frame eigenvalues")
print("DPsim state count:", modal_dpsim_check_a["state_count"])
print("Analytical mixed-frame state count:", analytic_mixed_check_a["Ad"].shape[0])
print("Maximum discrete eigenvalue mismatch |Δz|:", z_error_check_a)
print("Maximum continuous eigenvalue mismatch |Δlambda|:", lambda_error_check_a)

assert modal_dpsim_check_a["state_count"] == analytic_mixed_check_a["Ad"].shape[0]
assert z_error_check_a < 1e-8
assert lambda_error_check_a < 1e-4

print("Check A passed.")

In [ ]:
print("Check A rightmost eigenvalues sorted by decreasing Re(lambda):")

pd.concat(
    [
        eigenvalue_table(
            modal_dpsim_check_a["lambda"],
            "DPsim extracted",
            n=8,
        ),
        eigenvalue_table(
            analytic_mixed_check_a["lambda"],
            "Analytical mixed-frame",
            n=8,
        ),
    ],
    ignore_index=True,
)

## Check B: stability-classification consistency

This check compares the stability conclusion from three cases:

1. DPsim `GlobalDQ0` one-step discrete eigenvalues,
2. native-frame Floquet multipliers over one fundamental period,
3. analytical continuous-time global dq0 eigenvalues.

The analytical global dq0 model is obtained by applying a continuous-time coordinate transformation to the analytical mixed-frame model:

```text
xGlobalDq0 = T(t) xNative
```

which gives

```text
AGlobalDq0 = T ANative T^{-1} + Tdot T^{-1}.
```

The analytical model is used as a reference. The DPsim `GlobalDQ0` eigenvalues are computed from the extracted discrete simulation model. This check asserts only the stability classification.

In [7]:
def global_dq0_transform(theta):
    transform = np.eye(17)

    T3 = park_matrix_3(theta)

    transform[8:11, 8:11] = T3
    transform[11:14, 11:14] = T3
    transform[14:17, 14:17] = T3

    return transform


def global_dq0_transform_derivative(theta):
    transform_dot = np.zeros((17, 17))

    dT3 = park_matrix_3_derivative(theta)

    transform_dot[8:11, 8:11] = dT3
    transform_dot[11:14, 11:14] = dT3
    transform_dot[14:17, 14:17] = dT3

    return transform_dot


def analytical_global_dq0_model(op, gain_scale, theta_global=0.0):
    mixed = analytical_mixed_model(op, gain_scale)

    T = global_dq0_transform(theta_global)
    T_dot = omega0 * global_dq0_transform_derivative(theta_global)

    # The Park transform is power-invariant, so T^{-1} = T.T.
    T_inv = T.T

    A = T @ mixed["A"] @ T_inv + T_dot @ T_inv

    return {
        "A": A,
        "lambda": np.linalg.eigvals(A),
    }

## Modal, Floquet, and plotting helpers

In [8]:
def load_log(sim_name):
    return pd.read_csv(
        Path("logs") / sim_name / f"{sim_name}.csv",
        skipinitialspace=True,
    )


def signal(df, name):
    cols = [c for c in df.columns if c == name or c.startswith(name + "_")]

    return df[cols].to_numpy()


def scalar_signal(df, name):
    return signal(df, name)[:, 0]


def time_vector(df):
    return df.iloc[:, 0].to_numpy()


def is_discrete_stable(values):
    return bool(np.max(np.abs(finite_complex(values))) < 1.0)


def is_continuous_stable(values):
    return bool(np.max(finite_complex(values).real) < 0.0)


def floquet_from_ad_sequence(ad_sequence, period):
    Phi = np.eye(ad_sequence[0].shape[0])

    for Ad in ad_sequence:
        Phi = Ad @ Phi

    mu = np.linalg.eigvals(Phi)
    growth_rates = np.log(np.abs(mu)) / period

    return {
        "Phi": Phi,
        "mu": mu,
        "growth_rates": growth_rates,
    }


def run_modal_case(gain_scale, case_name, skip_periods=1):
    period = 1.0 / system_frequency
    steps_per_period = int(round(period / time_step))
    total_steps = (skip_periods + 1) * steps_per_period

    sim_name = f"{sim_name_base}_{case_name}_modal_gain_{gain_scale:g}".replace(
        ".", "_"
    )
    Logger.set_log_dir("logs/" + sim_name)

    op = steady_state_operating_point(gain_scale)
    system, inverter = build_dpsim_system(op, gain_scale)

    sim = Simulation(sim_name)
    sim.set_system(system)

    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)

    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)
    sim.do_state_space_extraction(True)

    sim.set_time_step(time_step)
    sim.set_final_time(total_steps * time_step)

    ad_over_period = []
    global_dq0_modal = None

    sim.start()

    for step_idx in range(total_steps):
        sim.next()

        if step_idx >= skip_periods * steps_per_period:
            if global_dq0_modal is None:
                global_dq0_modal = get_global_dq0_modal_results(sim, theta0=0.0)

            ad_over_period.append(get_native_modal_results(sim)["Ad"])

    sim.stop()

    if len(ad_over_period) != steps_per_period:
        raise RuntimeError("Unexpected number of state matrices collected.")

    if global_dq0_modal is None:
        raise RuntimeError("GlobalDQ0 modal results were not collected.")

    floquet = floquet_from_ad_sequence(
        ad_over_period,
        steps_per_period * time_step,
    )

    analytical_dq0 = analytical_global_dq0_model(
        op,
        gain_scale,
        theta_global=0.0,
    )

    mu_floquet = finite_complex(floquet["mu"])
    z_global_dq0 = finite_complex(global_dq0_modal["z"])
    lambda_global_dq0 = finite_complex(global_dq0_modal["lambda"])
    lambda_analytical_dq0 = finite_complex(analytical_dq0["lambda"])

    return {
        "gain_scale": gain_scale,
        "op": op,
        "steps_per_period": steps_per_period,
        "period": steps_per_period * time_step,
        "mu_floquet": mu_floquet,
        "z_global_dq0": z_global_dq0,
        "lambda_global_dq0": lambda_global_dq0,
        "lambda_analytical_dq0": lambda_analytical_dq0,
        "max_abs_mu_floquet": np.max(np.abs(mu_floquet)),
        "max_abs_z_global_dq0": np.max(np.abs(z_global_dq0)),
        "max_abs_z_global_dq0_over_period": np.max(
            np.abs(z_global_dq0) ** steps_per_period
        ),
        "max_floquet_growth_rate": np.max(floquet["growth_rates"]),
        "max_re_lambda_global_dq0": np.max(lambda_global_dq0.real),
        "max_re_lambda_analytical_dq0": np.max(lambda_analytical_dq0.real),
        "stable_floquet": is_discrete_stable(mu_floquet),
        "stable_global_dq0": is_discrete_stable(z_global_dq0),
        "stable_analytical_dq0": is_continuous_stable(lambda_analytical_dq0),
    }


def run_dpsim_time_domain(gain_scale, case_name):
    return run_dpsim(
        gain_scale=gain_scale,
        final_time=final_time_validation,
        case_name=case_name,
        log_signals=True,
    )


def plot_unit_circle_eigenvalues(ax, values, title, label):
    values = finite_complex(values)

    phi = np.linspace(0.0, 2.0 * np.pi, 500)

    ax.plot(
        np.cos(phi),
        np.sin(phi),
        linestyle="--",
        linewidth=0.8,
        label="unit circle",
    )

    ax.scatter(values.real, values.imag, s=18, label=label)

    ax.axhline(0.0, linewidth=0.8)
    ax.axvline(0.0, linewidth=0.8)

    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True)
    ax.legend()


def plot_continuous_eigenvalue_comparison(
    ax,
    lambda_global_dq0,
    lambda_analytical_dq0,
    title,
):
    global_dq0 = finite_complex(lambda_global_dq0)
    analytical_dq0 = finite_complex(lambda_analytical_dq0)

    ax.scatter(
        analytical_dq0.real,
        analytical_dq0.imag,
        marker="x",
        label="analytical global dq0",
    )

    ax.scatter(
        global_dq0.real,
        global_dq0.imag,
        marker="o",
        facecolors="none",
        edgecolors="C1",
        label="DPsim GlobalDQ0",
    )

    ax.axhline(0.0, linewidth=0.8)
    ax.axvline(0.0, linewidth=0.8)

    ax.set_xlabel("Re(λ) [1/s]")
    ax.set_ylabel("Im(λ) [rad/s]")
    ax.set_title(title)
    ax.grid(True)
    ax.legend()


def plot_active_power_response(ax, df, title):
    t = time_vector(df)
    p = scalar_signal(df, "p_inst")

    (p_line,) = ax.plot(t, p, label="DPsim p_inst")

    ax.axhline(
        p_ref,
        linestyle="--",
        color=p_line.get_color(),
        label="p_ref",
    )

    ax.set_ylim(p_ref * 0.99, p_ref * 1.01)

    ax.set_xlabel("time [s]")
    ax.set_ylabel("active power [W]")
    ax.set_title(title)
    ax.ticklabel_format(axis="y", style="plain", useOffset=False)
    ax.grid(True)
    ax.legend()

In [ ]:
modal_stable = run_modal_case(
    gain_scale=gain_scale_stable,
    case_name="check_b_stable",
)

modal_unstable = run_modal_case(
    gain_scale=gain_scale_unstable,
    case_name="check_b_unstable",
)

modal_cases = [
    ("stable", modal_stable, True),
    ("unstable", modal_unstable, False),
]

for case_name, result, expected_stable in modal_cases:
    assert result["stable_global_dq0"] == expected_stable
    assert result["stable_floquet"] == expected_stable
    assert result["stable_analytical_dq0"] == expected_stable

print("Check B passed.")
print("Steps per period:", modal_stable["steps_per_period"])

pd.DataFrame(
    [
        {
            "case": case_name,
            "gain scale": result["gain_scale"],
            "GlobalDQ0 stable": result["stable_global_dq0"],
            "Floquet stable": result["stable_floquet"],
            "Analytical dq0 stable": result["stable_analytical_dq0"],
            "max |z|, DPsim GlobalDQ0": result["max_abs_z_global_dq0"],
            "max |z|^N, DPsim GlobalDQ0": result["max_abs_z_global_dq0_over_period"],
            "max |mu|, native Floquet": result["max_abs_mu_floquet"],
            "max Re(lambda), DPsim GlobalDQ0 [1/s]": result["max_re_lambda_global_dq0"],
            "max Re(lambda), analytical dq0 [1/s]": result[
                "max_re_lambda_analytical_dq0"
            ],
        }
        for case_name, result, expected_stable in modal_cases
    ]
)

## Check C: modal and time-domain demonstration near the stability boundary

This check visualizes the stable and unstable cases close to the stability boundary.

For each case, the plots show:

1. native-frame Floquet multipliers over one period,
2. DPsim `GlobalDQ0` one-step discrete eigenvalues,
3. DPsim `GlobalDQ0` continuous eigenvalues calculated from the discrete eigenvalues and compared with analytical continuous-time global dq0 eigenvalues,
4. DPsim EMT active-power response.

The purpose of this check is to demonstrate that the DPsim `GlobalDQ0` modal results give the same stability conclusion as the native-frame Floquet multipliers and are consistent with the observed EMT simulation response.

In [ ]:
dpsim_stable = run_dpsim_time_domain(
    gain_scale=gain_scale_stable,
    case_name="check_c_stable",
)

dpsim_unstable = run_dpsim_time_domain(
    gain_scale=gain_scale_unstable,
    case_name="check_c_unstable",
)

df_stable = load_log(dpsim_stable["sim_name"])
df_unstable = load_log(dpsim_unstable["sim_name"])

pd.DataFrame(
    [
        {
            "case": "stable",
            "gain scale": modal_stable["gain_scale"],
            "max |mu|, native Floquet": modal_stable["max_abs_mu_floquet"],
            "max |z|, DPsim GlobalDQ0": modal_stable["max_abs_z_global_dq0"],
            "max Re(lambda), DPsim GlobalDQ0 [1/s]": modal_stable[
                "max_re_lambda_global_dq0"
            ],
            "max Re(lambda), analytical dq0 [1/s]": modal_stable[
                "max_re_lambda_analytical_dq0"
            ],
        },
        {
            "case": "unstable",
            "gain scale": modal_unstable["gain_scale"],
            "max |mu|, native Floquet": modal_unstable["max_abs_mu_floquet"],
            "max |z|, DPsim GlobalDQ0": modal_unstable["max_abs_z_global_dq0"],
            "max Re(lambda), DPsim GlobalDQ0 [1/s]": modal_unstable[
                "max_re_lambda_global_dq0"
            ],
            "max Re(lambda), analytical dq0 [1/s]": modal_unstable[
                "max_re_lambda_analytical_dq0"
            ],
        },
    ]
)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(21, 9))

plot_unit_circle_eigenvalues(
    axes[0, 0],
    modal_stable["mu_floquet"],
    f"Native Floquet multipliers, gain = {gain_scale_stable:g}",
    "native Floquet μ",
)

plot_unit_circle_eigenvalues(
    axes[0, 1],
    modal_stable["z_global_dq0"],
    f"GlobalDQ0 discrete eigenvalues, gain = {gain_scale_stable:g}",
    "DPsim GlobalDQ0 z",
)

plot_continuous_eigenvalue_comparison(
    axes[0, 2],
    modal_stable["lambda_global_dq0"],
    modal_stable["lambda_analytical_dq0"],
    f"Continuous eigenvalues, gain = {gain_scale_stable:g}",
)

plot_active_power_response(
    axes[0, 3],
    df_stable,
    f"DPsim EMT simulation, gain = {gain_scale_stable:g}",
)

plot_unit_circle_eigenvalues(
    axes[1, 0],
    modal_unstable["mu_floquet"],
    f"Native Floquet multipliers, gain = {gain_scale_unstable:g}",
    "native Floquet μ",
)

plot_unit_circle_eigenvalues(
    axes[1, 1],
    modal_unstable["z_global_dq0"],
    f"GlobalDQ0 discrete eigenvalues, gain = {gain_scale_unstable:g}",
    "DPsim GlobalDQ0 z",
)

plot_continuous_eigenvalue_comparison(
    axes[1, 2],
    modal_unstable["lambda_global_dq0"],
    modal_unstable["lambda_analytical_dq0"],
    f"Continuous eigenvalues, gain = {gain_scale_unstable:g}",
)

plot_active_power_response(
    axes[1, 3],
    df_unstable,
    f"DPsim EMT simulation, gain = {gain_scale_unstable:g}",
)

plt.tight_layout()
plt.show()